# SAKETIME 日本酒ランキング 探索的データ分析

データソース: https://www.saketime.jp/ranking/

## 目次
1. データ読み込み・概要
2. 評価スコアの分布
3. レビュー数の分布
4. 評価とレビュー数の関係
5. 都道府県別分析（銘柄詳細データ取得後）
6. 価格帯分析（銘柄詳細データ取得後）
7. テイスト分析（レビューデータ取得後）

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
import seaborn as sns
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore', message='Glyph.*missing from font')

# スタイル適用後にフォントを設定（スタイルがフォント設定をオーバーライドするため）
plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Hiragino Sans', 'Hiragino Maru Gothic Pro', 'YuGothic', 'Arial']
matplotlib.rcParams['axes.unicode_minus'] = False

DATA_DIR = '../data'
print('準備完了')

## 1. データ読み込み・概要

In [ ]:
# ランキングデータ読み込み
ranking = pd.read_csv(os.path.join(DATA_DIR, 'ranking.csv'))
print(f'銘柄数: {len(ranking):,}')
print(f'カラム: {list(ranking.columns)}')
print(f'\n基本統計量:')
ranking[['rating', 'review_count']].describe()

In [ ]:
# 上位20銘柄
ranking[['rank', 'name', 'rating', 'review_count']].head(20)

## 2. 評価スコアの分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ヒストグラム
axes[0].hist(ranking['rating'], bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('評価スコア')
axes[0].set_ylabel('銘柄数')
axes[0].set_title('評価スコアの分布')
axes[0].axvline(ranking['rating'].mean(), color='red', linestyle='--', label=f'平均: {ranking["rating"].mean():.2f}')
axes[0].axvline(ranking['rating'].median(), color='green', linestyle='--', label=f'中央値: {ranking["rating"].median():.2f}')
axes[0].legend()

# 箱ひげ図
axes[1].boxplot(ranking['rating'], vert=True)
axes[1].set_ylabel('評価スコア')
axes[1].set_title('評価スコアの箱ひげ図')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'rating_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. レビュー数の分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# レビュー数のヒストグラム（対数スケール）
axes[0].hist(ranking['review_count'], bins=100, color='#FF9800', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('レビュー数')
axes[0].set_ylabel('銘柄数')
axes[0].set_title('レビュー数の分布')
axes[0].set_yscale('log')

# レビュー数の対数ヒストグラム
log_reviews = np.log10(ranking['review_count'])
axes[1].hist(log_reviews, bins=50, color='#FF9800', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('log10(レビュー数)')
axes[1].set_ylabel('銘柄数')
axes[1].set_title('レビュー数の分布（対数スケール）')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'review_count_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'レビュー数上位10銘柄:')
ranking.nlargest(10, 'review_count')[['rank', 'name', 'rating', 'review_count']]

## 4. 評価とレビュー数の関係

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    ranking['review_count'], ranking['rating'],
    alpha=0.3, s=10, c=ranking['rating'], cmap='RdYlGn'
)
ax.set_xlabel('レビュー数')
ax.set_ylabel('評価スコア')
ax.set_title('評価スコア vs レビュー数')
ax.set_xscale('log')
plt.colorbar(scatter, label='評価スコア')

# 上位銘柄にラベル
for _, row in ranking.head(10).iterrows():
    ax.annotate(row['name'], (row['review_count'], row['rating']),
                fontsize=8, alpha=0.8, 
                xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'rating_vs_reviews.png'), dpi=150, bbox_inches='tight')
plt.show()

# 相関係数
corr = ranking['rating'].corr(ranking['review_count'])
corr_log = ranking['rating'].corr(np.log10(ranking['review_count']))
print(f'相関係数（評価 vs レビュー数）: {corr:.3f}')
print(f'相関係数（評価 vs log(レビュー数)）: {corr_log:.3f}')

In [ ]:
# レビュー数の区間別の平均評価
bins = [0, 5, 10, 25, 50, 100, 250, 500, 1000, 10000]
labels = ['1-5', '6-10', '11-25', '26-50', '51-100', '101-250', '251-500', '501-1000', '1001+']
ranking['review_bin'] = pd.cut(ranking['review_count'], bins=bins, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 区間別銘柄数
review_counts = ranking['review_bin'].value_counts().sort_index()
review_counts.plot(kind='bar', ax=axes[0], color='#2196F3', edgecolor='white')
axes[0].set_xlabel('レビュー数区間')
axes[0].set_ylabel('銘柄数')
axes[0].set_title('レビュー数区間別の銘柄数')
axes[0].tick_params(axis='x', rotation=45)

# 区間別平均評価
mean_rating = ranking.groupby('review_bin')['rating'].mean()
mean_rating.plot(kind='bar', ax=axes[1], color='#4CAF50', edgecolor='white')
axes[1].set_xlabel('レビュー数区間')
axes[1].set_ylabel('平均評価スコア')
axes[1].set_title('レビュー数区間別の平均評価')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(2.8, 4.2)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'review_bins_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. 説明文のテキスト基本分析

In [ ]:
# 説明文があるデータのみ
has_desc = ranking.dropna(subset=['description'])
print(f'説明文あり: {len(has_desc)}銘柄 / {len(ranking)}銘柄 ({len(has_desc)/len(ranking)*100:.1f}%)')

# 説明文の文字数分布
has_desc = has_desc.copy()
has_desc['desc_length'] = has_desc['description'].str.len()
print(f'説明文の文字数統計:')
print(has_desc['desc_length'].describe())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(has_desc['desc_length'], bins=50, color='#9C27B0', edgecolor='white', alpha=0.8)
ax.set_xlabel('説明文の文字数')
ax.set_ylabel('銘柄数')
ax.set_title('説明文の文字数分布')
plt.tight_layout()
plt.show()

## 6. 都道府県別分析（銘柄詳細データ）

※ brands.csvが生成された後に実行

In [ ]:
brands_file = os.path.join(DATA_DIR, 'brands.csv')
if os.path.exists(brands_file):
    brands = pd.read_csv(brands_file)
    print(f'銘柄詳細データ: {len(brands)}件')
    
    # ランキングと結合
    df = ranking.merge(brands, on='brand_id', how='left')
    print(f'結合後: {len(df)}件')
    print(f'都道府県データあり: {df["prefecture"].notna().sum()}件')
    
    # 都道府県別の銘柄数
    pref_counts = df['prefecture'].value_counts()
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 12))
    
    # 都道府県別銘柄数
    pref_counts.head(20).plot(kind='barh', ax=axes[0], color='#2196F3')
    axes[0].set_xlabel('銘柄数')
    axes[0].set_title('都道府県別 銘柄数 Top 20')
    axes[0].invert_yaxis()
    
    # 都道府県別平均評価（10銘柄以上）
    pref_rating = df.groupby('prefecture').agg(
        mean_rating=('rating', 'mean'),
        count=('rating', 'count')
    ).query('count >= 10').sort_values('mean_rating', ascending=False)
    
    pref_rating['mean_rating'].head(20).plot(kind='barh', ax=axes[1], color='#4CAF50')
    axes[1].set_xlabel('平均評価スコア')
    axes[1].set_title('都道府県別 平均評価 Top 20（10銘柄以上）')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, 'prefecture_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('brands.csv がまだ生成されていません。brand_scraper.pyの実行完了を待ってください。')

In [ ]:
# 価格帯分析
if os.path.exists(brands_file):
    df_price = df.dropna(subset=['min_price'])
    print(f'価格情報あり: {len(df_price)}銘柄')
    
    if len(df_price) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 最低価格の分布
        axes[0].hist(df_price['min_price'], bins=50, color='#FF9800', edgecolor='white', alpha=0.8)
        axes[0].set_xlabel('最低価格（円）')
        axes[0].set_ylabel('銘柄数')
        axes[0].set_title('最低価格の分布')
        
        # 価格と評価の関係
        axes[1].scatter(df_price['min_price'], df_price['rating'], alpha=0.3, s=10)
        axes[1].set_xlabel('最低価格（円）')
        axes[1].set_ylabel('評価スコア')
        axes[1].set_title('価格 vs 評価')
        axes[1].set_xscale('log')
        
        plt.tight_layout()
        plt.savefig(os.path.join(DATA_DIR, 'price_analysis.png'), dpi=150, bbox_inches='tight')
        plt.show()
        
        corr = df_price['min_price'].corr(df_price['rating'])
        print(f'価格と評価の相関: {corr:.3f}')

## 7. レビューデータ分析

※ reviews.csvが生成された後に実行

In [ ]:
reviews_file = os.path.join(DATA_DIR, 'reviews.csv')
if os.path.exists(reviews_file) and os.path.getsize(reviews_file) > 100:
    try:
        reviews = pd.read_csv(reviews_file)
        print(f'レビュー数: {len(reviews):,}')
        print(f'\nカラム: {list(reviews.columns)}')
        print(f'\n欠損値:')
        print(reviews.isnull().sum())
        print(f'\n評価点の統計:')
        print(reviews['rating'].describe())
    except Exception as e:
        print(f'reviews.csv読み込みエラー: {e}')
        reviews = None
else:
    print('reviews.csv がまだ生成されていません。review_scraper.pyの実行完了を待ってください。')
    reviews = None

In [ ]:
if 'reviews' in dir() and reviews is not None and len(reviews) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # レビュー評価の分布
    reviews['rating'].hist(bins=20, ax=axes[0, 0], color='#2196F3', edgecolor='white')
    axes[0, 0].set_xlabel('評価点')
    axes[0, 0].set_ylabel('レビュー数')
    axes[0, 0].set_title('レビュー評価点の分布')
    
    # 特定名称別の評価
    if 'specific_name' in reviews.columns:
        spec_rating = reviews.dropna(subset=['specific_name']).groupby('specific_name')['rating'].agg(['mean', 'count'])
        spec_rating = spec_rating[spec_rating['count'] >= 100].sort_values('mean', ascending=False)
        spec_rating['mean'].plot(kind='barh', ax=axes[0, 1], color='#4CAF50')
        axes[0, 1].set_xlabel('平均評価')
        axes[0, 1].set_title('特定名称別 平均評価（100件以上）')
        axes[0, 1].invert_yaxis()
    
    # ボディ別の評価
    if 'body' in reviews.columns:
        body_rating = reviews.dropna(subset=['body']).groupby('body')['rating'].agg(['mean', 'count'])
        body_order = ['軽い+2', '軽い+1', '普通', '重い+1', '重い+2']
        body_rating = body_rating.reindex([b for b in body_order if b in body_rating.index])
        body_rating['mean'].plot(kind='bar', ax=axes[1, 0], color='#FF9800')
        axes[1, 0].set_xlabel('ボディ')
        axes[1, 0].set_ylabel('平均評価')
        axes[1, 0].set_title('ボディ別 平均評価')
        axes[1, 0].tick_params(axis='x', rotation=45)
    
    # 甘辛別の評価
    if 'sweetness' in reviews.columns:
        sweet_rating = reviews.dropna(subset=['sweetness']).groupby('sweetness')['rating'].agg(['mean', 'count'])
        sweet_order = ['辛い+2', '辛い+1', '普通', '甘い+1', '甘い+2']
        sweet_rating = sweet_rating.reindex([s for s in sweet_order if s in sweet_rating.index])
        sweet_rating['mean'].plot(kind='bar', ax=axes[1, 1], color='#E91E63')
        axes[1, 1].set_xlabel('甘辛')
        axes[1, 1].set_ylabel('平均評価')
        axes[1, 1].set_title('甘辛別 平均評価')
        axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, 'review_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('レビューデータ未取得のためスキップ')

In [ ]:
# テイストマッピング（ボディ x 甘辛）
if 'reviews' in dir() and reviews is not None and 'body' in reviews.columns and 'sweetness' in reviews.columns:
    taste_df = reviews.dropna(subset=['body', 'sweetness'])
    print(f'テイストデータあり: {len(taste_df):,}件')
    
    body_order = ['軽い+2', '軽い+1', '普通', '重い+1', '重い+2']
    sweet_order = ['辛い+2', '辛い+1', '普通', '甘い+1', '甘い+2']
    
    cross = pd.crosstab(taste_df['body'], taste_df['sweetness'])
    cross = cross.reindex(index=[b for b in body_order if b in cross.index],
                          columns=[s for s in sweet_order if s in cross.columns])
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cross, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
    ax.set_xlabel('甘辛')
    ax.set_ylabel('ボディ')
    ax.set_title('テイストマッピング（ボディ × 甘辛）')
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, 'taste_mapping.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('レビューデータ未取得のためスキップ')

In [ ]:
# 原料米分析
if 'reviews' in dir() and reviews is not None and 'rice' in reviews.columns:
    rice_data = reviews.dropna(subset=['rice'])
    print(f'原料米データあり: {len(rice_data):,}件')
    
    rice_counts = rice_data['rice'].value_counts()
    print(f'原料米の種類: {len(rice_counts)}')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    rice_counts.head(20).plot(kind='barh', ax=axes[0], color='#8BC34A')
    axes[0].set_xlabel('使用回数')
    axes[0].set_title('原料米 使用頻度 Top 20')
    axes[0].invert_yaxis()
    
    rice_rating = rice_data.groupby('rice')['rating'].agg(['mean', 'count'])
    rice_rating = rice_rating[rice_rating['count'] >= 100].sort_values('mean', ascending=False)
    rice_rating['mean'].head(20).plot(kind='barh', ax=axes[1], color='#FF5722')
    axes[1].set_xlabel('平均評価')
    axes[1].set_title('原料米別 平均評価 Top 20（100件以上）')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, 'rice_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('レビューデータ未取得のためスキップ')

In [ ]:
# 時系列分析（レビュー日付）
if 'reviews' in dir() and reviews is not None:
    reviews_ts = reviews.dropna(subset=['date']).copy()
    import re
    def parse_date(d):
        m = re.match(r'(\d{4})年(\d{1,2})月(\d{1,2})日', str(d))
        if m:
            return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        return pd.NaT
    
    reviews_ts['parsed_date'] = reviews_ts['date'].apply(parse_date)
    reviews_ts = reviews_ts.dropna(subset=['parsed_date'])
    print(f'日付パース成功: {len(reviews_ts):,}件')
    print(f'期間: {reviews_ts["parsed_date"].min()} ~ {reviews_ts["parsed_date"].max()}')
    
    monthly = reviews_ts.set_index('parsed_date').resample('M').size()
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    monthly.plot(ax=axes[0], color='#2196F3')
    axes[0].set_xlabel('月')
    axes[0].set_ylabel('レビュー数')
    axes[0].set_title('月別レビュー投稿数の推移')
    
    monthly_rating = reviews_ts.set_index('parsed_date').resample('M')['rating'].mean()
    monthly_rating.plot(ax=axes[1], color='#4CAF50')
    axes[1].set_xlabel('月')
    axes[1].set_ylabel('平均評価')
    axes[1].set_title('月別平均評価の推移')
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, 'timeseries_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('レビューデータ未取得のためスキップ')